In [ ]:
%%writefile stores.csv
store_id,store_name,city,state,store_type,manager_name
S101,Metro Mart Hyderabad,Hyderabad,Telangana,Supermarket,Rahul Sharma
S102,Metro Mart Bangalore,Bangalore,Karnataka,Supermarket,Priya Reddy
S103,Metro Mart Mumbai,Mumbai,Maharashtra,Hypermarket,Amit Kumar
S104,Metro Mart Chennai,Chennai,Tamil Nadu,Supermarket,Sneha Patel
S105,Metro Mart Delhi,Delhi,Delhi,Hypermarket,Farhan Ali
S106,Metro Mart Pune,Pune,Maharashtra,Mini Store,Neha Singh
S107,Metro Mart Kochi,Kochi,Kerala,Mini Store,Arjun Verma
S108,Metro Mart Jaipur,Jaipur,Rajasthan,Supermarket,Meera Nair

In [ ]:
%%writefile products.csv
product_id,product_name,category,brand,supplier_id,unit_price
P101,Laptop,Electronics,Lenovo,S201,65000
P102,Mobile,Electronics,Samsung,S202,25000
P103,Television,Electronics,LG,S203,45000
P104,Office Chair,Furniture,Featherlite,S204,7000
P105,Study Table,Furniture,Urban Ladder,S204,12000
P106,Shoes,Fashion,Nike,S205,4500
P107,Watch,Fashion,Fastrack,S206,8000
P108,Backpack,Fashion,Wildcraft,S206,2500
P109,Refrigerator,Electronics,Whirlpool,S203,38000
P110,Sofa,Furniture,Godrej,S204,32000
P111,Headphones,Electronics,Sony,S999,3000
P112,T-Shirt,Fashion,Puma,,1500

In [ ]:
%%writefile inventory.csv
inventory_id,store_id,product_id,stock_quantity,reorder_level,last_update
I1001,S101,P101,10,5,2026-01-10
I1002,S101,P102,25,10,2026-01-10
I1003,S101,P104,3,5,2026-01-11
I1004,S102,P101,8,5,2026-01-12
I1005,S102,P103,5,4,2026-01-12
I1006,S103,P105,2,5,2026-01-13
I1007,S103,P106,30,10,2026-01-14
I1008,S104,P107,4,5,2026-01-15
I1009,S105,P108,50,20,2026-01-15
I1010,S106,P109,,6,2026-01-16
I1011,S107,P110,1,3,2026-01-17
I1012,S108,P120,12,5,2026-01-18

In [ ]:
%%writefile sales.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_mode
SA1001,S101,P101,2026-01-10,1,65000,UPI
SA1002,S101,P102,2026-01-10,2,50000,Card
SA1003,S102,P101,2026-01-11,1,65000,UPI
SA1004,S103,P106,2026-01-12,4,18000,Cash
SA1005,S104,P107,2026-01-12,1,8000,Card
SA1006,S105,P108,2026-01-13,5,12500,UPI
SA1007,S106,P109,2026-01-14,1,38000,Card
SA1008,S107,P110,2026-01-15,1,32000,UPI
SA1009,S108,P120,2026-01-15,2,10000,Cash
SA1010,S101,P104,2026-01-16,2,14000,
SA1011,S102,P103,2026-01-17,1,,UPI
SA1012,S103,P105,2026-01-18,1,12000,Card
SA1013,S104,P107,2026-02-01,2,16000,UPI
SA1014,S105,P108,2026-02-02,3,7500,Cash
SA1015,S101,P102,2026-02-03,1,25000,Card

In [ ]:
%%writefile suppliers.json
[
 {
  "supplier_id": "S201",
  "supplier_name": "TechSource India",
  "city": "Hyderabad",
  "rating": 4.5,
  "contact": {
   "phone": "9876500011",
   "email": "techsource@mail.com"
  }
 },
 {
  "supplier_id": "S202",
  "supplier_name": "MobileWorld Distributors",
  "city": "Bangalore",
  "rating": 4.2,
  "contact": {
   "phone": null,
   "email": "mobileworld@mail.com"
  }
 },
 {
  "supplier_id": "S203",
  "supplier_name": "HomeTech Supply",
  "city": "Mumbai",
  "rating": 4.4,
  "contact": {
   "phone": "9876500013",
   "email": null
  }
 },
 {
  "supplier_id": "S204",
  "supplier_name": "Urban Furniture Co",
  "city": "Delhi",
  "rating": 4.0,
  "contact": {
   "phone": "9876500014",
   "email": "urban@mail.com"
  }
 },
 {
  "supplier_id": "S205",
  "supplier_name": "Fashion Direct",
  "city": "Pune",
  "rating": 3.8,
  "contact": {
   "phone": null,
   "email": null
  }
 }
]

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("RetailCapstone").getOrCreate()

In [7]:
stores_df = spark.read.csv(
    "stores.csv",
    header=True,
    inferSchema=True
)

stores_df.show()

+--------+--------------------+---------+-----------+-----------+------------+
|store_id|          store_name|     city|      state| store_type|manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya Reddy|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|Hypermarket|  Amit Kumar|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|Supermarket| Sneha Patel|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|Hypermarket|  Farhan Ali|
|    S106|     Metro Mart Pune|     Pune|Maharashtra| Mini Store|  Neha Singh|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala| Mini Store| Arjun Verma|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|Supermarket|  Meera Nair|
+--------+--------------------+---------+-----------+-----------+------------+



In [8]:
products_df = spark.read.csv(
    "products.csv",
    header=True,
    inferSchema=True
)

products_df.show()

+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P103|  Television|Electronics|          LG|       S203|     45000|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|
|      P110|        Sofa|  Furniture|      Godrej|       S204|     32000|
|      P111|  Headphones|Electronics| 

In [9]:
inventory_df = spark.read.csv(
    "inventory.csv",
    header=True,
    inferSchema=True
)

inventory_df.show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|
|       I1009|    S105|      P108|            50|           20| 2026-01-15|
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
|       I101

In [10]:
sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|      UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|     Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|      UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|     Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|     Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|      UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|     Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|      UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|     Cash|
| SA1010|    S101|      P104|2026-01-16|            2|      14000|     NULL|

In [12]:
with open("suppliers.json", "r") as f:
    print(f.read())

[
{
"supplier_id": "S201",
"supplier_name": "TechSource India",
"city": "Hyderabad",
"rating": 4.5,
"contact": {
"phone": "9876500011",
"email": "techsource@mail.com"
}
},
{
"supplier_id": "S202",
"supplier_name": "MobileWorld Distributors",
"city": "Bangalore",
"rating": 4.2,
"contact": {
"phone": null,
"email": "mobileworld@mail.com"
%%writefile sales.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_m
SA1001,S101,P101,2026-01-10,1,65000,UPI
SA1002,S101,P102,2026-01-10,2,50000,Card
SA1003,S102,P101,2026-01-11,1,65000,UPI
SA1004,S103,P106,2026-01-12,4,18000,Cash
SA1005,S104,P107,2026-01-12,1,8000,Card
SA1006,S105,P108,2026-01-13,5,12500,UPI
SA1007,S106,P109,2026-01-14,1,38000,Card
SA1008,S107,P110,2026-01-15,1,32000,UPI
SA1009,S108,P120,2026-01-15,2,10000,Cash
SA1010,S101,P104,2026-01-16,2,14000,
SA1011,S102,P103,2026-01-17,1,,UPI
SA1012,S103,P105,2026-01-18,1,12000,Card
SA1013,S104,P107,2026-02-01,2,16000,UPI
SA1014,S105,P108,2026-02-02,3,7500,Cash
SA1015,S1

In [18]:
suppliers_df = spark.read.option("multiline", "true").json("suppliers.json")
suppliers_df.show(truncate=False)

+---------+---------------------------------+------+-----------+------------------------+
|city     |contact                          |rating|supplier_id|supplier_name           |
+---------+---------------------------------+------+-----------+------------------------+
|Hyderabad|{techsource@mail.com, 9876500011}|4.5   |S201       |TechSource India        |
|Bangalore|{mobileworld@mail.com, NULL}     |4.2   |S202       |MobileWorld Distributors|
+---------+---------------------------------+------+-----------+------------------------+



In [19]:
stores_df = spark.read.csv(
    "stores.csv",
    header=True,
    inferSchema=True
)

stores_df.show()

+--------+--------------------+---------+-----------+-----------+------------+
|store_id|          store_name|     city|      state| store_type|manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya Reddy|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|Hypermarket|  Amit Kumar|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|Supermarket| Sneha Patel|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|Hypermarket|  Farhan Ali|
|    S106|     Metro Mart Pune|     Pune|Maharashtra| Mini Store|  Neha Singh|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala| Mini Store| Arjun Verma|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|Supermarket|  Meera Nair|
+--------+--------------------+---------+-----------+-----------+------------+



In [20]:
print("Stores Schema")
stores_df.printSchema()

print("Products Schema")
products_df.printSchema()

print("Inventory Schema")
inventory_df.printSchema()

print("Sales Schema")
sales_df.printSchema()

print("Suppliers Schema")
suppliers_df.printSchema()

Stores Schema
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

Products Schema
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

Inventory Schema
root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

Sales Schema
root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- qu

In [21]:
print("Stores Count:", stores_df.count())

print("Products Count:", products_df.count())

print("Inventory Count:", inventory_df.count())

print("Sales Count:", sales_df.count())

print("Suppliers Count:", suppliers_df.count())

Stores Count: 8
Products Count: 12
Inventory Count: 12
Sales Count: 15
Suppliers Count: 2


In [22]:
stores_df.write.mode("overwrite").parquet("bronze/stores")

products_df.write.mode("overwrite").parquet("bronze/products")

inventory_df.write.mode("overwrite").parquet("bronze/inventory")

sales_df.write.mode("overwrite").parquet("bronze/sales")

suppliers_df.write.mode("overwrite").parquet("bronze/suppliers")

In [24]:
products_df.filter(
    col("supplier_id").isNull() | (col("supplier_id") == "")
).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [25]:
inventory_df.filter(
    col("stock_quantity").isNull()
).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [26]:
sales_df.filter(
    col("sale_amount").isNull()
).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|      UPI|
+-------+--------+----------+----------+-------------+-----------+---------+



In [30]:
sales_df.columns

['sale_id',
 'store_id',
 'product_id',
 'sale_date',
 'quantity_sold',
 'sale_amount',
 'payment_m']

In [31]:
sales_df = sales_df.withColumnRenamed("payment_m", "payment_mode")

In [32]:
sales_df.printSchema()

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- payment_mode: string (nullable = true)



In [ ]:

sales_df.filter(
    col("payment_mode").isNull() | (col("payment_mode") == "")
).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|        NULL|
+-------+--------+----------+----------+-------------+-----------+------------+



In [34]:
inventory_df = inventory_df.fillna({
    "stock_quantity": 0
})

inventory_df.show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|
|       I1009|    S105|      P108|            50|           20| 2026-01-15|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|
|       I101

In [35]:
sales_df = sales_df.fillna({
    "sale_amount": 0
})

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010|    S101|      P104|2026-01-16|

In [36]:
sales_df = sales_df.fillna({
    "payment_mode": "Not Provided"
})

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010|    S101|      P104|2026-01-16|

In [37]:
products_df = products_df.fillna({
    "supplier_id": "UNKNOWN"
})

products_df.show()

+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P103|  Television|Electronics|          LG|       S203|     45000|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|
|      P110|        Sofa|  Furniture|      Godrej|       S204|     32000|
|      P111|  Headphones|Electronics| 

In [38]:
products_df = products_df.fillna({
    "supplier_id": "UNKNOWN"
})

products_df.show()

+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P103|  Television|Electronics|          LG|       S203|     45000|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|
|      P110|        Sofa|  Furniture|      Godrej|       S204|     32000|
|      P111|  Headphones|Electronics| 

In [39]:
products_df = products_df.withColumn(
    "data_quality_status",
    when(col("supplier_id") == "UNKNOWN", "Incomplete")
    .otherwise("Complete")
)

products_df.show()

+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|           Complete|
|      P103|  Television|Electronics|          LG|       S203|     45000|           Complete|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|           Complete|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|           Complete|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|           Complete|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|           Complete|
|      P108|    Backpack|    Fashion|   Wildcraft|       S20

In [40]:
products_df.write.mode("overwrite").parquet("silver/products")

inventory_df.write.mode("overwrite").parquet("silver/inventory")

sales_df.write.mode("overwrite").parquet("silver/sales")

In [41]:
from pyspark.sql.functions import col

In [42]:
suppliers_flat_df = suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

suppliers_flat_df.show(truncate=False)

+-----------+------------------------+---------+------+----------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone     |email               |
+-----------+------------------------+---------+------+----------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011|techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |NULL      |mobileworld@mail.com|
+-----------+------------------------+---------+------+----------+--------------------+



In [43]:
suppliers_flat_df.select(
    "supplier_id",
    "phone"
).show(truncate=False)

+-----------+----------+
|supplier_id|phone     |
+-----------+----------+
|S201       |9876500011|
|S202       |NULL      |
+-----------+----------+



In [44]:
suppliers_flat_df.select(
    "supplier_id",
    "email"
).show(truncate=False)

+-----------+--------------------+
|supplier_id|email               |
+-----------+--------------------+
|S201       |techsource@mail.com |
|S202       |mobileworld@mail.com|
+-----------+--------------------+



In [45]:
suppliers_flat_df = suppliers_flat_df.fillna({
    "phone": "Not Provided"
})

suppliers_flat_df.show(truncate=False)

+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
+-----------+------------------------+---------+------+------------+--------------------+



In [46]:
suppliers_flat_df = suppliers_flat_df.fillna({
    "email": "Not Provided"
})

suppliers_flat_df.show(truncate=False)

+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
+-----------+------------------------+---------+------+------------+--------------------+



In [47]:
suppliers_flat_df.write.mode("overwrite").parquet("silver/suppliers_flat")

In [48]:
from pyspark.sql.functions import col

In [49]:
products_suppliers_df = products_df.join(
    suppliers_flat_df,
    on="supplier_id",
    how="left"
)

products_suppliers_df.show(truncate=False)

+-----------+----------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|supplier_id|product_id|product_name|category   |brand       |unit_price|data_quality_status|supplier_name           |city     |rating|phone       |email               |
+-----------+----------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|S201       |P101      |Laptop      |Electronics|Lenovo      |65000     |Complete           |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |P102      |Mobile      |Electronics|Samsung     |25000     |Complete           |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |P103      |Television  |Electronics|LG          |45000     |Complete           |NULL                    |NULL     |NULL  |NULL        |NU

In [50]:
inventory_products_df = inventory_df.join(
    products_df,
    on="product_id",
    how="left"
)

inventory_products_df.show(truncate=False)

+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|category   |brand       |supplier_id|unit_price|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+-------------------+
|P101      |I1001       |S101    |10            |5            |2026-01-10 |Laptop      |Electronics|Lenovo      |S201       |65000     |Complete           |
|P102      |I1002       |S101    |25            |10           |2026-01-10 |Mobile      |Electronics|Samsung     |S202       |25000     |Complete           |
|P104      |I1003       |S101    |3             |5            |2026-01-11 |Office Chair|Furniture  |Featherlite |S204       |7000      |Complete           |
|P101      |I1004       |S102    |8             |5        

In [51]:
sales_stores_df = sales_df.join(
    stores_df,
    on="store_id",
    how="left"
)

sales_stores_df.show(truncate=False)

+--------+-------+----------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|store_name          |city     |state      |store_type |manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+
|S101    |SA1001 |P101      |2026-01-10|1            |65000      |UPI         |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S101    |SA1002 |P102      |2026-01-10|2            |50000      |Card        |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S102    |SA1003 |P101      |2026-01-11|1            |65000      |UPI         |Metro Mart Bangalore|Bangalore|Karnataka  |Supermarket|Priya Reddy |
|S103    |SA1004 |P106      |2026-01-12|4            |18000      |Cash        |Metro Mart Mumbai   |Mumbai   |Ma

In [52]:
sales_products_df = sales_df.join(
    products_df,
    on="product_id",
    how="left"
)

sales_products_df.show(truncate=False)

+----------+-------+--------+----------+-------------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|sale_id|store_id|sale_date |quantity_sold|sale_amount|payment_mode|product_name|category   |brand       |supplier_id|unit_price|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|P101      |SA1001 |S101    |2026-01-10|1            |65000      |UPI         |Laptop      |Electronics|Lenovo      |S201       |65000     |Complete           |
|P102      |SA1002 |S101    |2026-01-10|2            |50000      |Card        |Mobile      |Electronics|Samsung     |S202       |25000     |Complete           |
|P101      |SA1003 |S102    |2026-01-11|1            |65000      |UPI         |Laptop      |Electronics|Lenovo      |S201       |65000     |Complete           |
|P106      |SA1004 |S103    |2026-

In [53]:
retail_sales_df = sales_df \
    .join(stores_df, "store_id", "left") \
    .join(products_df, "product_id", "left") \
    .join(suppliers_flat_df, "supplier_id", "left")

retail_sales_df.show(truncate=False)

+-----------+----------+--------+-------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|supplier_id|product_id|store_id|sale_id|sale_date |quantity_sold|sale_amount|payment_mode|store_name          |city     |state      |store_type |manager_name|product_name|category   |brand       |unit_price|data_quality_status|supplier_name           |city     |rating|phone       |email               |
+-----------+----------+--------+-------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|S201       |P101      |S101    |SA1001 |2026-01-10|1            |65000      |UPI    

In [54]:
products_df.join(
    suppliers_flat_df,
    on="supplier_id",
    how="left_anti"
).show()

+-----------+----------+------------+-----------+------------+----------+-------------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+------------+----------+-------------------+
|       S203|      P103|  Television|Electronics|          LG|     45000|           Complete|
|       S204|      P104|Office Chair|  Furniture| Featherlite|      7000|           Complete|
|       S204|      P105| Study Table|  Furniture|Urban Ladder|     12000|           Complete|
|       S205|      P106|       Shoes|    Fashion|        Nike|      4500|           Complete|
|       S206|      P107|       Watch|    Fashion|    Fastrack|      8000|           Complete|
|       S206|      P108|    Backpack|    Fashion|   Wildcraft|      2500|           Complete|
|       S203|      P109|Refrigerator|Electronics|   Whirlpool|     38000|           Complete|
|       S204|      P110|        Sofa|  Furniture|      Godre

In [55]:
inventory_df.join(
    products_df,
    on="product_id",
    how="left_anti"
).show()

+----------+------------+--------+--------------+-------------+-----------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|
+----------+------------+--------+--------------+-------------+-----------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|
+----------+------------+--------+--------------+-------------+-----------+



In [56]:
sales_df.join(
    products_df,
    on="product_id",
    how="left_anti"
).show()

+----------+-------+--------+----------+-------------+-----------+------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_mode|
+----------+-------+--------+----------+-------------+-----------+------------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|        Cash|
+----------+-------+--------+----------+-------------+-----------+------------+



In [57]:
sales_df.join(
    stores_df,
    on="store_id",
    how="left_anti"
).show()

+--------+-------+----------+---------+-------------+-----------+------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_mode|
+--------+-------+----------+---------+-------------+-----------+------------+
+--------+-------+----------+---------+-------------+-----------+------------+



In [58]:
from pyspark.sql.functions import col, when, month, year

In [59]:
inventory_df = inventory_df.withColumn(
    "stock_status",
    when(
        col("stock_quantity") <= col("reorder_level"),
        "Reorder Required"
    ).otherwise("Sufficient Stock")
)

inventory_df.show()

+------------+--------+----------+--------------+-------------+-----------+----------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|    stock_status|
+------------+--------+----------+--------------+-------------+-----------+----------------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|Sufficient Stock|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|Sufficient Stock|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|Reorder Required|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|Sufficient Stock|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|Sufficient Stock|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|Reorder Required|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|Sufficient Stock|
|       I1008|    S104|      P107|             4|            5| 2026-0

In [60]:
products_df = products_df.withColumn(
    "price_category",
    when(col("unit_price") >= 50000, "Premium")
    .when(col("unit_price") >= 10000, "Standard")
    .otherwise("Budget")
)

products_df.show()

+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|       Premium|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|           Complete|      Standard|
|      P103|  Television|Electronics|          LG|       S203|     45000|           Complete|      Standard|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|           Complete|        Budget|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|           Complete|      Standard|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|           Complete|        Budget|
|      P107|       

In [61]:
sales_df = sales_df.withColumn(
    "revenue_category",
    when(col("sale_amount") >= 50000, "High Revenue")
    .when(col("sale_amount") >= 15000, "Medium Revenue")
    .otherwise("Low Revenue")
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|    High Revenue|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|    High Revenue|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|    High Revenue|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|  Medium Revenue|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|     Low Revenue|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|     Low Revenue|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|  Medium Revenue|
| SA1008|    S107|      P110|2

In [62]:
sales_df = sales_df.withColumn(
    "month",
    month(col("sale_date"))
)

sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|    High Revenue|    1|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|    High Revenue|    1|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|    High Revenue|    1|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|  Medium Revenue|    1|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|     Low Revenue|    1|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|     Low Revenue|    1|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|       

In [63]:
sales_df = sales_df.withColumn(
    "year",
    year(col("sale_date"))
)

sales_df.show()


+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|     Low Revenue|    1|2026|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|     Low Revenue|    1|2026|
| SA1007|    S106|      P109

In [64]:
inventory_value_df = inventory_df.join(
    products_df.select("product_id", "unit_price"),
    on="product_id",
    how="left"
)

inventory_value_df = inventory_value_df.withColumn(
    "inventory_value",
    col("stock_quantity") * col("unit_price")
)

inventory_value_df.show()

+----------+------------+--------+--------------+-------------+-----------+----------------+----------+---------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|    stock_status|unit_price|inventory_value|
+----------+------------+--------+--------------+-------------+-----------+----------------+----------+---------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|Sufficient Stock|     65000|         650000|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|Sufficient Stock|     25000|         625000|
|      P104|       I1003|    S101|             3|            5| 2026-01-11|Reorder Required|      7000|          21000|
|      P101|       I1004|    S102|             8|            5| 2026-01-12|Sufficient Stock|     65000|         520000|
|      P103|       I1005|    S102|             5|            4| 2026-01-12|Sufficient Stock|     45000|         225000|
|      P105|       I1006|    S103|      

In [65]:
suppliers_flat_df = suppliers_flat_df.withColumn(
    "supplier_quality",
    when(col("rating") >= 4.5, "Excellent")
    .when(col("rating") >= 4.0, "Good")
    .otherwise("Average")
)

suppliers_flat_df.show()

+-----------+--------------------+---------+------+------------+--------------------+----------------+
|supplier_id|       supplier_name|     city|rating|       phone|               email|supplier_quality|
+-----------+--------------------+---------+------+------------+--------------------+----------------+
|       S201|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|       Excellent|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|            Good|
+-----------+--------------------+---------+------+------------+--------------------+----------------+



In [66]:
from pyspark.sql.functions import col, count, sum, desc

In [67]:
stores_df.groupBy("state") \
    .count() \
    .show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
|Maharashtra|    2|
+-----------+-----+



In [68]:
products_df.groupBy("brand") \
    .count() \
    .show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [69]:
products_df.groupBy("brand") \
    .count() \
    .show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [70]:
inventory_value_df.groupBy("store_id") \
    .agg(sum("inventory_value").alias("total_inventory_value")) \
    .show()

+--------+---------------------+
|store_id|total_inventory_value|
+--------+---------------------+
|    S105|               125000|
|    S102|               745000|
|    S106|                    0|
|    S104|                32000|
|    S107|                32000|
|    S101|              1296000|
|    S108|                 NULL|
|    S103|               159000|
+--------+---------------------+



In [71]:
inventory_value_df.join(
    products_df.select("product_id", "category"),
    "product_id"
).groupBy("category") \
 .agg(sum("inventory_value").alias("total_inventory_value")) \
 .show()

+-----------+---------------------+
|   category|total_inventory_value|
+-----------+---------------------+
|    Fashion|               292000|
|Electronics|              2020000|
|  Furniture|                77000|
+-----------+---------------------+



In [72]:
inventory_df.filter(
    col("stock_status") == "Reorder Required"
).count()

5

In [73]:
sales_df.agg(
    sum("sale_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|       373000|
+-------------+



In [74]:
sales_df.groupBy("store_id") \
    .agg(sum("sale_amount").alias("total_revenue")) \
    .show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S105|        20000|
|    S102|        65000|
|    S106|        38000|
|    S104|        24000|
|    S107|        32000|
|    S101|       154000|
|    S108|        10000|
|    S103|        30000|
+--------+-------------+



In [75]:
sales_df.join(
    stores_df,
    "store_id"
).groupBy("city") \
 .agg(sum("sale_amount").alias("total_revenue")) \
 .show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|        65000|
|    Kochi|        32000|
|  Chennai|        24000|
|   Mumbai|        30000|
|     Pune|        38000|
|    Delhi|        20000|
|Hyderabad|       154000|
|   Jaipur|        10000|
+---------+-------------+



In [76]:
sales_df.join(
    products_df,
    "product_id"
).groupBy("category") \
 .agg(sum("sale_amount").alias("total_revenue")) \
 .show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|    Fashion|        62000|
|Electronics|       243000|
|  Furniture|        58000|
+-----------+-------------+



In [77]:
sales_df.join(
    products_df,
    "product_id"
).groupBy("product_name") \
 .agg(sum("sale_amount").alias("total_revenue")) \
 .show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|Office Chair|        14000|
|Refrigerator|        38000|
|      Laptop|       130000|
|        Sofa|        32000|
|    Backpack|        20000|
|       Shoes|        18000|
|      Mobile|        75000|
|  Television|            0|
| Study Table|        12000|
|       Watch|        24000|
+------------+-------------+



In [78]:
sales_df.groupBy("payment_mode") \
    .agg(sum("sale_amount").alias("total_revenue")) \
    .show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
|         UPI|       190500|
+------------+-------------+



In [79]:
sales_df.join(
    products_df,
    "product_id"
).groupBy("product_name") \
 .agg(sum("sale_amount").alias("total_revenue")) \
 .orderBy(desc("total_revenue")) \
 .limit(1) \
 .show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|      Laptop|       130000|
+------------+-------------+



In [80]:
sales_df.join(
    stores_df,
    "store_id"
).groupBy("store_name") \
 .agg(sum("sale_amount").alias("total_revenue")) \
 .orderBy(desc("total_revenue")) \
 .limit(1) \
 .show()

+--------------------+-------------+
|          store_name|total_revenue|
+--------------------+-------------+
|Metro Mart Hyderabad|       154000|
+--------------------+-------------+



In [81]:
sales_df.join(
    products_df,
    "product_id"
).groupBy("category") \
 .agg(sum("sale_amount").alias("total_revenue")) \
 .orderBy(desc("total_revenue")) \
 .limit(1) \
 .show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       243000|
+-----------+-------------+



In [82]:
from pyspark.sql.functions import sum, col, lag, lead
from pyspark.sql.window import Window

In [84]:
from pyspark.sql.functions import rank

In [85]:
product_revenue = sales_df.join(
    products_df, "product_id"
).groupBy(
    "product_id", "product_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

window_spec = Window.orderBy(col("total_revenue").desc())

product_revenue.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+----------+------------+-------------+----+
|product_id|product_name|total_revenue|rank|
+----------+------------+-------------+----+
|      P101|      Laptop|       130000|   1|
|      P102|      Mobile|        75000|   2|
|      P109|Refrigerator|        38000|   3|
|      P110|        Sofa|        32000|   4|
|      P107|       Watch|        24000|   5|
|      P108|    Backpack|        20000|   6|
|      P106|       Shoes|        18000|   7|
|      P104|Office Chair|        14000|   8|
|      P105| Study Table|        12000|   9|
|      P103|  Television|            0|  10|
+----------+------------+-------------+----+



In [86]:
category_revenue = sales_df.join(
    products_df, "product_id"
).groupBy(
    "category",
    "product_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

window_spec = Window.partitionBy(
    "category"
).orderBy(
    col("total_revenue").desc()
)

category_revenue.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      Laptop|       130000|   1|
|Electronics|      Mobile|        75000|   2|
|Electronics|Refrigerator|        38000|   3|
|Electronics|  Television|            0|   4|
|    Fashion|       Watch|        24000|   1|
|    Fashion|    Backpack|        20000|   2|
|    Fashion|       Shoes|        18000|   3|
|  Furniture|        Sofa|        32000|   1|
|  Furniture|Office Chair|        14000|   2|
|  Furniture| Study Table|        12000|   3|
+-----------+------------+-------------+----+



In [87]:
category_revenue = sales_df.join(
    products_df, "product_id"
).groupBy(
    "category",
    "product_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

window_spec = Window.partitionBy(
    "category"
).orderBy(
    col("total_revenue").desc()
)

category_revenue.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") == 1
).show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      Laptop|       130000|   1|
|    Fashion|       Watch|        24000|   1|
|  Furniture|        Sofa|        32000|   1|
+-----------+------------+-------------+----+



In [88]:
category_revenue.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") <= 3
).show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      Laptop|       130000|   1|
|Electronics|      Mobile|        75000|   2|
|Electronics|Refrigerator|        38000|   3|
|    Fashion|       Watch|        24000|   1|
|    Fashion|    Backpack|        20000|   2|
|    Fashion|       Shoes|        18000|   3|
|  Furniture|        Sofa|        32000|   1|
|  Furniture|Office Chair|        14000|   2|
|  Furniture| Study Table|        12000|   3|
+-----------+------------+-------------+----+



In [89]:
state_revenue = sales_df.join(
    stores_df, "store_id"
).groupBy(
    "state",
    "store_name"
).agg(
    sum("sale_amount").alias("total_revenue")
)

window_spec = Window.partitionBy(
    "state"
).orderBy(
    col("total_revenue").desc()
)

state_revenue.withColumn(
    "rank",
    rank().over(window_spec)
).filter(
    col("rank") == 1
).show()

+-----------+--------------------+-------------+----+
|      state|          store_name|total_revenue|rank|
+-----------+--------------------+-------------+----+
|      Delhi|    Metro Mart Delhi|        20000|   1|
|  Karnataka|Metro Mart Bangalore|        65000|   1|
|     Kerala|    Metro Mart Kochi|        32000|   1|
|Maharashtra|     Metro Mart Pune|        38000|   1|
|  Rajasthan|   Metro Mart Jaipur|        10000|   1|
| Tamil Nadu|  Metro Mart Chennai|        24000|   1|
|  Telangana|Metro Mart Hyderabad|       154000|   1|
+-----------+--------------------+-------------+----+



In [90]:
daily_sales = sales_df.groupBy(
    "sale_date"
).agg(
    sum("sale_amount").alias("daily_revenue")
)

window_spec = Window.orderBy("sale_date")

daily_sales.withColumn(
    "running_total",
    sum("daily_revenue").over(window_spec)
).show()

+----------+-------------+-------------+
| sale_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2026-01-10|       115000|       115000|
|2026-01-11|        65000|       180000|
|2026-01-12|        26000|       206000|
|2026-01-13|        12500|       218500|
|2026-01-14|        38000|       256500|
|2026-01-15|        42000|       298500|
|2026-01-16|        14000|       312500|
|2026-01-17|            0|       312500|
|2026-01-18|        12000|       324500|
|2026-02-01|        16000|       340500|
|2026-02-02|         7500|       348000|
|2026-02-03|        25000|       373000|
+----------+-------------+-------------+



In [91]:
daily_sales = sales_df.groupBy(
    "sale_date"
).agg(
    sum("sale_amount").alias("daily_revenue")
)

window_spec = Window.orderBy("sale_date")

daily_sales.withColumn(
    "previous_day_sales",
    lag("daily_revenue", 1).over(window_spec)
).show()

+----------+-------------+------------------+
| sale_date|daily_revenue|previous_day_sales|
+----------+-------------+------------------+
|2026-01-10|       115000|              NULL|
|2026-01-11|        65000|            115000|
|2026-01-12|        26000|             65000|
|2026-01-13|        12500|             26000|
|2026-01-14|        38000|             12500|
|2026-01-15|        42000|             38000|
|2026-01-16|        14000|             42000|
|2026-01-17|            0|             14000|
|2026-01-18|        12000|                 0|
|2026-02-01|        16000|             12000|
|2026-02-02|         7500|             16000|
|2026-02-03|        25000|              7500|
+----------+-------------+------------------+



In [92]:
window_spec = Window.orderBy("sale_date")

sales_df.withColumn(
    "next_sale_amount",
    lead("sale_amount", 1).over(window_spec)
).show()

+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|next_sale_amount|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|    High Revenue|    1|2026|           50000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|    High Revenue|    1|2026|           65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|    High Revenue|    1|2026|           18000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|  Medium Revenue|    1|2026|            8000|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|     Low Revenue|    1|2026|           12500|


In [93]:
product_sales = sales_df.join(
    products_df, "product_id"
)

window_spec = Window.partitionBy(
    "product_id"
).orderBy(
    "sale_date"
)

product_sales = product_sales.withColumn(
    "previous_sale_amount",
    lag("sale_amount", 1).over(window_spec)
)

product_sales.filter(
    col("sale_amount") > col("previous_sale_amount")
).select(
    "product_id",
    "product_name",
    "sale_date",
    "previous_sale_amount",
    "sale_amount"
).show()

+----------+------------+----------+--------------------+-----------+
|product_id|product_name| sale_date|previous_sale_amount|sale_amount|
+----------+------------+----------+--------------------+-----------+
|      P107|       Watch|2026-02-01|                8000|      16000|
+----------+------------+----------+--------------------+-----------+



In [94]:
stores_df.createOrReplaceTempView("stores")

products_df.createOrReplaceTempView("products")

inventory_df.createOrReplaceTempView("inventory")

sales_df.createOrReplaceTempView("sales")

suppliers_flat_df.createOrReplaceTempView("suppliers")

In [95]:
spark.sql("""
SELECT
    category,
    COUNT(*) AS product_count
FROM products
GROUP BY category
ORDER BY product_count DESC
""").show()

+-----------+-------------+
|   category|product_count|
+-----------+-------------+
|Electronics|            5|
|    Fashion|            4|
|  Furniture|            3|
+-----------+-------------+



In [96]:
spark.sql("""
SELECT *
FROM sales
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|     Low Revenue|    1|2026|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|     Low Revenue|    1|2026|
| SA1007|    S106|      P109

In [97]:
spark.sql("""
SELECT
    category,
    COUNT(*) AS product_count
FROM products
GROUP BY category
ORDER BY product_count DESC
""").show()

+-----------+-------------+
|   category|product_count|
+-----------+-------------+
|Electronics|            5|
|    Fashion|            4|
|  Furniture|            3|
+-----------+-------------+



In [98]:
spark.sql("""
SELECT
    st.store_name,
    SUM(sa.sale_amount) AS total_revenue
FROM sales sa
JOIN stores st
ON sa.store_id = st.store_id
GROUP BY st.store_name
ORDER BY total_revenue DESC
""").show()

+--------------------+-------------+
|          store_name|total_revenue|
+--------------------+-------------+
|Metro Mart Hyderabad|       154000|
|Metro Mart Bangalore|        65000|
|     Metro Mart Pune|        38000|
|    Metro Mart Kochi|        32000|
|   Metro Mart Mumbai|        30000|
|  Metro Mart Chennai|        24000|
|    Metro Mart Delhi|        20000|
|   Metro Mart Jaipur|        10000|
+--------------------+-------------+



In [99]:
spark.sql("""
SELECT
    st.city,
    SUM(sa.sale_amount) AS total_revenue
FROM sales sa
JOIN stores st
ON sa.store_id = st.store_id
GROUP BY st.city
ORDER BY total_revenue DESC
""").show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|       154000|
|Bangalore|        65000|
|     Pune|        38000|
|    Kochi|        32000|
|   Mumbai|        30000|
|  Chennai|        24000|
|    Delhi|        20000|
|   Jaipur|        10000|
+---------+-------------+



In [100]:
spark.sql("""
SELECT *
FROM inventory
WHERE stock_quantity <= reorder_level
""").show()

+------------+--------+----------+--------------+-------------+-----------+----------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|    stock_status|
+------------+--------+----------+--------------+-------------+-----------+----------------+
|       I1003|    S101|      P104|             3|            5| 2026-01-11|Reorder Required|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|Reorder Required|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|Reorder Required|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|Reorder Required|
|       I1011|    S107|      P110|             1|            3| 2026-01-17|Reorder Required|
+------------+--------+----------+--------------+-------------+-----------+----------------+



In [101]:
spark.sql("""
SELECT s.*
FROM sales s
LEFT JOIN products p
ON s.product_id = p.product_id
WHERE p.product_id IS NULL
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|     Low Revenue|    1|2026|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+



In [102]:
spark.sql("""
SELECT p.*
FROM products p
LEFT JOIN suppliers s
ON p.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""").show()

+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|      P103|  Television|Electronics|          LG|       S203|     45000|           Complete|      Standard|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|           Complete|        Budget|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|           Complete|      Standard|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|           Complete|        Budget|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|           Complete|        Budget|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|           Complete|        Budget|
|      P109|Refrige

In [103]:
spark.sql("""
SELECT
    p.product_name,
    SUM(s.sale_amount) AS total_revenue
FROM sales s
JOIN products p
ON s.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_revenue DESC
LIMIT 5
""").show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|      Laptop|       130000|
|      Mobile|        75000|
|Refrigerator|        38000|
|        Sofa|        32000|
|       Watch|        24000|
+------------+-------------+



In [104]:
spark.sql("""
SELECT
    payment_mode,
    SUM(sale_amount) AS total_revenue
FROM sales
GROUP BY payment_mode
ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|         UPI|       190500|
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
+------------+-------------+



In [105]:
spark.sql("""
SELECT
    payment_mode,
    SUM(sale_amount) AS total_revenue
FROM sales
GROUP BY payment_mode
ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|         UPI|       190500|
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
+------------+-------------+



In [107]:
print(retail_sales_df.columns)

['supplier_id', 'product_id', 'store_id', 'sale_id', 'sale_date', 'quantity_sold', 'sale_amount', 'payment_mode', 'store_name', 'city', 'state', 'store_type', 'manager_name', 'product_name', 'category', 'brand', 'unit_price', 'data_quality_status', 'supplier_name', 'city', 'rating', 'phone', 'email']


In [108]:
from pyspark.sql.functions import col

suppliers_new_df = suppliers_flat_df.withColumnRenamed(
    "city", "supplier_city"
)

retail_sales_df = sales_df \
    .join(stores_df, "store_id", "left") \
    .join(products_df, "product_id", "left") \
    .join(suppliers_new_df, "supplier_id", "left")

retail_sales_df.show()

+-----------+----------+--------+-------+----------+-------------+-----------+------------+----------------+-----+----+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------+--------------------+-------------+------+------------+--------------------+----------------+
|supplier_id|product_id|store_id|sale_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|          store_name|     city|      state| store_type|manager_name|product_name|   category|       brand|unit_price|data_quality_status|price_category|       supplier_name|supplier_city|rating|       phone|               email|supplier_quality|
+-----------+----------+--------+-------+----------+-------------+-----------+------------+----------------+-----+----+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+-------------------+--------------

In [109]:
retail_sales_df.write \
    .mode("overwrite") \
    .parquet("gold/retail_sales")

In [110]:
retail_sales_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("gold/retail_sales_partitioned")

In [111]:
march_sales_df = sales_df.filter(
    (col("year") == 2026) &
    (col("month") == 3)
)

march_sales_df.write.mode("overwrite").parquet("incremental/march_2026_sales")

In [112]:
incremental_sales_df = spark.read.parquet(
    "incremental/march_2026_sales"
)

incremental_sales_df.show()

+-------+--------+----------+---------+-------------+-----------+------------+----------------+-----+----+
|sale_id|store_id|product_id|sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|
+-------+--------+----------+---------+-------------+-----------+------------+----------------+-----+----+
+-------+--------+----------+---------+-------------+-----------+------------+----------------+-----+----+



In [113]:
incremental_sales_df.write \
    .mode("append") \
    .parquet("silver/sales")

In [114]:
silver_sales_df = spark.read.parquet("silver/sales")

silver_sales_df.join(
    products_df,
    "product_id"
).groupBy(
    "product_name"
).agg(
    sum("sale_amount").alias("total_revenue")
).show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|Office Chair|        14000|
|Refrigerator|        38000|
|      Laptop|       130000|
|        Sofa|        32000|
|    Backpack|        20000|
|       Shoes|        18000|
|      Mobile|        75000|
|  Television|            0|
| Study Table|        12000|
|       Watch|        24000|
+------------+-------------+



In [115]:
silver_sales_df.join(
    stores_df,
    "store_id"
).groupBy(
    "store_name"
).agg(
    sum("sale_amount").alias("total_revenue")
).show()

+--------------------+-------------+
|          store_name|total_revenue|
+--------------------+-------------+
|Metro Mart Bangalore|        65000|
|    Metro Mart Kochi|        32000|
|   Metro Mart Jaipur|        10000|
|   Metro Mart Mumbai|        30000|
|     Metro Mart Pune|        38000|
|Metro Mart Hyderabad|       154000|
|    Metro Mart Delhi|        20000|
|  Metro Mart Chennai|        24000|
+--------------------+-------------+



In [116]:
gold_df = silver_sales_df \
    .join(stores_df, "store_id", "left") \
    .join(products_df, "product_id", "left")

gold_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("gold/final_retail_sales")

In [117]:
before_count = sales_df.count()

after_count = spark.read.parquet(
    "silver/sales"
).count()

print("Count Before Incremental Load :", before_count)
print("Count After Incremental Load  :", after_count)
print("New Records Added             :", after_count - before_count)

Count Before Incremental Load : 15
Count After Incremental Load  : 15
New Records Added             : 0


In [118]:
from pyspark.sql.functions import sum, count, col

In [119]:
store_performance_df = sales_df \
    .join(stores_df, "store_id") \
    .groupBy(
        "store_id",
        "store_name",
        "city",
        "state"
    ) \
    .agg(
        count("sale_id").alias("total_sales"),
        sum("sale_amount").alias("total_revenue")
    )

store_performance_df.show()

+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|          store_name|     city|      state|total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|          1|        38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|          1|        32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|          4|       154000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|          2|        30000|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|          2|        20000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|          2|        65000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|          2|        24000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|          1|        10000|
+--------+--------------------+---------+-----------+-----------+-------------+



In [120]:
product_performance_df = sales_df \
    .join(products_df, "product_id") \
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "brand"
    ) \
    .agg(
        sum("quantity_sold").alias("total_quantity_sold"),
        sum("sale_amount").alias("total_revenue")
    )

product_performance_df.show()

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|   category|       brand|total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|      P110|        Sofa|  Furniture|      Godrej|                  1|        32000|
|      P104|Office Chair|  Furniture| Featherlite|                  2|        14000|
|      P101|      Laptop|Electronics|      Lenovo|                  2|       130000|
|      P109|Refrigerator|Electronics|   Whirlpool|                  1|        38000|
|      P105| Study Table|  Furniture|Urban Ladder|                  1|        12000|
|      P107|       Watch|    Fashion|    Fastrack|                  3|        24000|
|      P102|      Mobile|Electronics|     Samsung|                  3|        75000|
|      P108|    Backpack|    Fashion|   Wildcraft|                  8|        20000|
|      P106|       Shoes|    Fashion|        Nike|               

In [121]:
inventory_reorder_df = inventory_df \
    .join(
        products_df.select(
            "product_id",
            "product_name"
        ),
        "product_id"
    ) \
    .select(
        "store_id",
        "product_id",
        "product_name",
        "stock_quantity",
        "reorder_level",
        "stock_status"
    )

inventory_reorder_df.show()

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|    stock_status|
+--------+----------+------------+--------------+-------------+----------------+
|    S101|      P101|      Laptop|            10|            5|Sufficient Stock|
|    S101|      P102|      Mobile|            25|           10|Sufficient Stock|
|    S101|      P104|Office Chair|             3|            5|Reorder Required|
|    S102|      P101|      Laptop|             8|            5|Sufficient Stock|
|    S102|      P103|  Television|             5|            4|Sufficient Stock|
|    S103|      P105| Study Table|             2|            5|Reorder Required|
|    S103|      P106|       Shoes|            30|           10|Sufficient Stock|
|    S104|      P107|       Watch|             4|            5|Reorder Required|
|    S105|      P108|    Backpack|            50|           20|Sufficient Stock|
|    S106|      P109|Refrige

In [122]:
supplier_quality_df = suppliers_flat_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    "supplier_quality",
    "phone",
    "email"
)

supplier_quality_df.show()

+-----------+--------------------+---------+------+----------------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|supplier_quality|       phone|               email|
+-----------+--------------------+---------+------+----------------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|       Excellent|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|            Good|Not Provided|mobileworld@mail.com|
+-----------+--------------------+---------+------+----------------+------------+--------------------+



In [123]:
category_revenue_df = sales_df \
    .join(products_df, "product_id") \
    .groupBy("category") \
    .agg(
        count("product_id").alias("total_products"),
        sum("quantity_sold").alias("total_quantity_sold"),
        sum("sale_amount").alias("total_revenue")
    )

category_revenue_df.show()

+-----------+--------------+-------------------+-------------+
|   category|total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|    Fashion|             5|                 15|        62000|
|Electronics|             6|                  7|       243000|
|  Furniture|             3|                  4|        58000|
+-----------+--------------+-------------------+-------------+



In [124]:
payment_mode_report_df = sales_df.groupBy(
    "payment_mode"
).agg(
    count("sale_id").alias("total_transactions"),
    sum("sale_amount").alias("total_revenue")
)

payment_mode_report_df.show()

+------------+------------------+-------------+
|payment_mode|total_transactions|total_revenue|
+------------+------------------+-------------+
|        Card|                 5|       133000|
|        Cash|                 3|        35500|
|Not Provided|                 1|        14000|
|         UPI|                 6|       190500|
+------------+------------------+-------------+

